# RISED Framework: Interactive Demo

**Reliability · Inclusivity · Sensitivity · Equity · Deployability**

This notebook demonstrates the RISED Framework on a 10,000-patient synthetic cohort
generated with a [Synthea](https://synthetichealth.github.io/synthea/)-inspired model.

> Paper: *Evaluating AI-Assisted Decision Support in High-Stakes Healthcare:*
> *A Framework for Reliability, Inclusivity, Sensitivity, Equity, and Deployability*
> (arXiv: ARXIV_ID_PLACEHOLDER)

**All patient data used here are entirely synthetic. No real patient records are used.**

In [ ]:
# Install the rised package (uncomment one line as needed)
# !pip install rised                                                          # PyPI release
# !pip install git+https://github.com/rohithreddybc/rised-healthcare-eval.git  # from source
# !pip install xgboost  # optional but recommended

In [ ]:
import rised
from rised.datasets import load_synthea_cohort, train_baseline_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pandas as pd

print(f"rised version: {rised.__version__}")

## 1. Load the Synthetic Cohort

The cohort contains 10,000 synthetic patients weighted toward Medicare/Medicaid enrollees
(older adults, higher comorbidity burden). If the CSV is not found locally, it is generated
on-the-fly via `generate_synthea_cohort()`.

In [ ]:
X, y, demographic_df = load_synthea_cohort()

print(f"Cohort shape:          {X.shape}")
print(f"High-need prevalence:  {y.mean():.1%}")
print(f"Feature columns:       {list(X.columns)}")
print(f"Demographic columns:   {list(demographic_df.columns)}")

In [ ]:
print("=== Age distribution ===")
print(demographic_df['age_group'].value_counts().sort_index())
print()
print("=== Race/ethnicity ===")
print(demographic_df['race'].value_counts())
print()
print("=== Insurance ===")
print(demographic_df['insurance'].value_counts())

## 2. Train / Test Split and Baseline Model

We use an 80/20 stratified split and train an XGBoost classifier as the baseline model.

In [ ]:
X_train, X_test, y_train, y_test, demo_train, demo_test = train_test_split(
    X, y, demographic_df,
    test_size=0.20, random_state=42, stratify=y
)

model = train_baseline_model(X_train, y_train)

probs_test = model.predict_proba(X_test)[:, 1]
auroc = roc_auc_score(y_test, probs_test)
print(f"Model type:   {type(model).__name__}")
print(f"Test AUROC:   {auroc:.4f}")
print(f"Train n:      {len(X_train)}  |  Test n: {len(X_test)}")

## 3. Run RISED Evaluation

`evaluate_all()` runs all five RISED dimensions in a single call and returns a
`FrameworkReport` containing per-dimension results and a summary.

In [ ]:
perturbation_specs = [
    {"type": "gaussian_noise", "scale": 0.05, "random_state": 0,  "label": "noise_5pct"},
    {"type": "gaussian_noise", "scale": 0.10, "random_state": 1,  "label": "noise_10pct"},
    {"type": "unit_rescaling", "feature_index": 0, "factor": 1.05, "label": "age_+5pct"},
]

report = rised.evaluate_all(
    model,
    X_test,
    y_test,
    demo_test,
    perturbation_specs=perturbation_specs,
    feature_names=list(X_test.columns),
)

print("RISED Evaluation complete.")
print(f"n_samples evaluated: {report.metadata['n_samples']}")

In [ ]:
print("=== RISED Framework Summary ===")
for dim, passed in report.summary().items():
    status = "PASS" if passed else "FAIL"
    print(f"  {dim:<20} {status}")
print()
print(f"All dimensions passed: {report.all_passed()}")

In [ ]:
r = report.reliability
i = report.inclusivity
s = report.sensitivity
e = report.equity
d = report.deployability

print("=== Reliability ===")
print(f"  Judge Sensitivity Score (JSS): {r.judge_sensitivity_score:.4f}  (pass < 0.05)")
print(f"  Mean flip rate:                {r.perturbation_flip_rate:.4f}")
print(f"  Mean rank correlation:         {r.rank_correlation_mean:.4f}")

print("\n=== Inclusivity ===")
print(f"  AUC parity gap:  {i.auc_parity_gap:.4f}  (pass <= 0.05)")
print(f"  Subgroup AUCs:   {i.subgroup_aucs}")

print("\n=== Sensitivity ===")
print(f"  Rank stability score:         {s.rank_stability_score:.4f}  (pass > 0.90)")
print(f"  Decision boundary width:      {s.decision_boundary_width:.4f}")

print("\n=== Equity ===")
print(f"  Need-prediction correlation:  {e.need_prediction_correlation:.4f}  (pass >= 0.70)")

print("\n=== Deployability ===")
print(f"  Mean inference latency:       {d.mean_inference_latency_ms:.2f} ms  (pass <= 500 ms)")
print(f"  Explanation faithfulness:     {d.explanation_faithfulness}")